# Validacao Dinamica de Performance no Fabric

Etapa 2 (CD) da esteira: roda **depois** que o Semantic Model chega ao Fabric via Git Integration e passa por um **refresh**.

O que este notebook faz:
1. Resolve o dataset por **nome ou id**.
2. **Estrutura e memoria** do modelo (model_memory_analyzer): tabelas, colunas, relacionamentos, particoes.
3. **Tamanho e incremental refresh** (tabelas import grandes sem policy).
4. **BPA no modelo publicado** (semantic-link-labs).
5. **Corretude de DAX**: valida se as medidas retornam o valor esperado.
6. **Performance**: mede o tempo de queries DAX (N execucoes, limpando cache).
7. **Health score + findings** consolidados (pode virar gate Dev -> Prod).

Pre-requisitos: modelo publicado + com refresh no workspace; ajuste `WORKSPACE`, `DATASET` e os testes de DAX as suas medidas.

> Ambiente controlado: sem guardrails de seguranca por enquanto (autenticacao usa o contexto do notebook).

In [ ]:
# Rode uma vez se necessario (ja vem no runtime recente do Fabric)
# %pip install semantic-link-labs

In [ ]:
import time
import pandas as pd
from datetime import datetime, timezone

import sempy.fabric as fabric
try:
    import sempy_labs as labs
except Exception as e:
    labs = None
    print('sempy_labs indisponivel:', e)

run_ts = datetime.now(timezone.utc)
findings = []

def add_finding(rule, severity, obj_type, obj_name, value=None, threshold=None, details=None):
    findings.append({'run_ts': run_ts, 'rule': rule, 'severity': severity,
                     'object_type': obj_type, 'object_name': obj_name,
                     'value': value, 'threshold': threshold, 'details': details})

def show(df):
    try:
        display(df)
    except Exception:
        print(df)

In [ ]:
# ===================== PARAMETROS =====================
WORKSPACE = 'SalesDemo Dev'      # nome ou id do workspace no Fabric
DATASET   = 'SalesDemo3'         # nome ou id do Semantic Model a validar

# Performance
PERF_RUNS = 5                    # execucoes por query para medir o tempo
CLEAR_CACHE_BETWEEN = True       # limpa o cache antes de cada execucao (cold cache)

# Limites (health)
CONFIG = {
    'model_size_mb_attention': 512,
    'model_size_mb_critical': 1024,               # 1 GB -> avaliar incremental
    'large_import_table_data_size_abs': 500000,   # bytes de Data Size
    'large_import_table_data_size_pct_model': 0.10,
    'perf_ms_attention': 1000,
    'perf_ms_critical': 3000,
}

# Testes de corretude (valor esperado das medidas). Ajuste aos seus dados.
DAX_CORRECTNESS_TESTS = [
    {'name': 'Total Revenue', 'dax': 'EVALUATE ROW("v", [Total Revenue])', 'expected': 17250, 'tol': 0.01},
    {'name': 'Total Units',   'dax': 'EVALUATE ROW("v", [Total Units])',   'expected': 700,   'tol': 0.01},
]

# Queries de performance (representam consumo tipico do report)
PERF_QUERIES = [
    {'name': 'Revenue by Product', 'dax': 'EVALUATE SUMMARIZECOLUMNS(Sales[Product], "Revenue", [Total Revenue])'},
    {'name': 'Revenue by Region',  'dax': 'EVALUATE SUMMARIZECOLUMNS(Sales[Region], "Revenue", [Total Revenue])'},
    {'name': 'Totais',             'dax': 'EVALUATE ROW("Rev", [Total Revenue], "Units", [Total Units])'},
]

GATE_FAIL_ON = {'High'}          # severidades que barram o gate
WRITE_TO_LAKEHOUSE = False       # True grava findings/summary em Delta (precisa lakehouse anexado)

In [ ]:
def resolve_dataset_id(dataset, workspace):
    try:
        dsets = fabric.list_datasets(workspace)
    except Exception as e:
        print('list_datasets falhou, usando valor informado:', e)
        return dataset
    name_col = next((c for c in dsets.columns if c.lower() in ('dataset name', 'name')), None)
    id_col = next((c for c in dsets.columns if c.lower() in ('dataset id', 'id')), None)
    if id_col and str(dataset) in dsets[id_col].astype(str).values:
        return str(dataset)
    if name_col and id_col:
        m = dsets[dsets[name_col].astype(str) == str(dataset)]
        if len(m):
            return str(m.iloc[0][id_col])
    return dataset

dataset_id = resolve_dataset_id(DATASET, WORKSPACE)
print('Dataset:', DATASET, '->', dataset_id, '| Workspace:', WORKSPACE)

## 1. Estrutura e memoria do modelo

In [ ]:
mem = fabric.model_memory_analyzer(dataset=dataset_id, workspace=WORKSPACE, return_dataframe=True)

tables = mem['Tables'].copy() if 'Tables' in mem else pd.DataFrame()
columns = mem['Columns'].copy() if 'Columns' in mem else pd.DataFrame()
relationships = mem['Relationships'].copy() if 'Relationships' in mem else pd.DataFrame()
partitions = mem['Partitions'].copy() if 'Partitions' in mem else pd.DataFrame()

if 'Model Summary' in mem:
    show(mem['Model Summary'])
show(tables)

## 2. Tamanho do modelo e incremental refresh

In [ ]:
def to_num(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce').fillna(0)
    return df

total_size_mb = 0.0
total_data_size = 0.0
if not tables.empty:
    tables = to_num(tables, ['Row Count', 'Total Size', 'Dictionary Size', 'Data Size'])
    total_size_mb = tables['Total Size'].sum() / 1000000
    total_data_size = tables['Data Size'].sum()
    print('Tamanho total do modelo: {:,.2f} MB'.format(total_size_mb))

    if total_size_mb >= CONFIG['model_size_mb_critical']:
        add_finding('MODEL_SIZE_CRITICAL', 'High', 'Dataset', DATASET, round(total_size_mb, 2), CONFIG['model_size_mb_critical'], 'Modelo grande: avalie incremental/otimizacao')
    elif total_size_mb >= CONFIG['model_size_mb_attention']:
        add_finding('MODEL_SIZE_ATTENTION', 'Medium', 'Dataset', DATASET, round(total_size_mb, 2), CONFIG['model_size_mb_attention'])

# Modo de particao por tabela (Import / DirectQuery / Mixed)
def resolve_modes(part_df):
    out = {}
    if part_df is None or part_df.empty or 'Mode' not in part_df.columns:
        return out
    p = part_df.copy()
    p['Mode'] = p['Mode'].astype(str)
    for tname, grp in p.groupby('Table Name'):
        modes = sorted(set(grp['Mode']))
        out[tname] = modes[0] if len(modes) == 1 else 'Mixed'
    return out

modes = resolve_modes(partitions)
if not tables.empty:
    tables['mode'] = tables['Table Name'].map(modes).fillna('Unknown')

# Incremental refresh apenas em tabela import considerada grande
tom = None
try:
    tom = fabric.TOMWrapper(dataset_id, WORKSPACE, readonly=True)
except Exception as e:
    print('TOMWrapper indisponivel:', e)

if tom is not None and not tables.empty:
    for _, row in tables.iterrows():
        is_import = str(row.get('mode', '')).lower() == 'import'
        data_size = row.get('Data Size', 0)
        is_large = data_size >= CONFIG['large_import_table_data_size_abs'] or (
            total_data_size > 0 and data_size / total_data_size >= CONFIG['large_import_table_data_size_pct_model'])
        if is_import and is_large:
            try:
                if not tom.has_incremental_refresh_policy(table_name=row['Table Name']):
                    add_finding('LARGE_IMPORT_WITHOUT_INCREMENTAL', 'High', 'Table', row['Table Name'], data_size, CONFIG['large_import_table_data_size_abs'], 'Tabela import grande sem incremental refresh')
            except Exception as e:
                add_finding('INCREMENTAL_CHECK_ERROR', 'Low', 'Table', row['Table Name'], details=str(e))

print('OK - checagem de tamanho/incremental concluida')

## 3. BPA no modelo publicado

In [ ]:
bpa = None
if labs is not None:
    try:
        bpa = labs.run_model_bpa(dataset=dataset_id, workspace=WORKSPACE, return_dataframe=True)
        show(bpa)
        sev_col = next((c for c in bpa.columns if c.lower() == 'severity'), None)
        name_col = next((c for c in bpa.columns if c.lower() in ('object name', 'object')), None)
        rule_col = next((c for c in bpa.columns if c.lower() in ('rule name', 'rule')), None)
        if sev_col:
            for _, b in bpa.iterrows():
                sev = str(b.get(sev_col, ''))
                if 'error' in sev.lower() or chr(9940) in sev:
                    add_finding('BPA_ERROR', 'High', 'Model', str(b.get(name_col, '')) if name_col else '', details=str(b.get(rule_col, '')) if rule_col else None)
    except Exception as e:
        print('run_model_bpa falhou:', e)
else:
    print('sempy_labs indisponivel: BPA pulado')

## 4. Corretude de DAX (valores esperados)

In [ ]:
def eval_scalar(dax):
    df = fabric.evaluate_dax(dataset=dataset_id, dax_string=dax, workspace=WORKSPACE)
    if df is None or len(df) == 0 or df.shape[1] == 0:
        return None
    return df.iloc[0, 0]

corr_rows = []
for t in DAX_CORRECTNESS_TESTS:
    try:
        val = eval_scalar(t['dax'])
        exp = t.get('expected')
        tol = t.get('tol', 0)
        ok = exp is None or (val is not None and abs(float(val) - float(exp)) <= abs(float(exp)) * tol + 1e-9)
        corr_rows.append({'test': t['name'], 'expected': exp, 'actual': val, 'passed': bool(ok)})
        if not ok:
            add_finding('DAX_CORRECTNESS_FAIL', 'High', 'Measure', t['name'], val, exp, 'Valor diferente do esperado')
    except Exception as e:
        corr_rows.append({'test': t['name'], 'expected': t.get('expected'), 'actual': None, 'passed': False})
        add_finding('DAX_EVAL_ERROR', 'High', 'Measure', t['name'], details=str(e))

correctness_df = pd.DataFrame(corr_rows)
show(correctness_df)

## 5. Testes de performance (tempo por query)

In [ ]:
def clear_cache():
    if labs is None or not CLEAR_CACHE_BETWEEN:
        return
    try:
        labs.clear_cache(dataset=dataset_id, workspace=WORKSPACE)
    except Exception:
        pass

perf_rows = []
for q in PERF_QUERIES:
    durs = []
    for _ in range(PERF_RUNS):
        clear_cache()
        t0 = time.perf_counter()
        try:
            _ = fabric.evaluate_dax(dataset=dataset_id, dax_string=q['dax'], workspace=WORKSPACE)
            durs.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            add_finding('PERF_QUERY_ERROR', 'Medium', 'Query', q['name'], details=str(e))
            durs = []
            break
    if durs:
        avg = sum(durs) / len(durs)
        perf_rows.append({'query': q['name'], 'runs': len(durs), 'avg_ms': round(avg, 1),
                          'min_ms': round(min(durs), 1), 'max_ms': round(max(durs), 1)})
        if avg >= CONFIG['perf_ms_critical']:
            add_finding('PERF_CRITICAL', 'High', 'Query', q['name'], round(avg, 1), CONFIG['perf_ms_critical'])
        elif avg >= CONFIG['perf_ms_attention']:
            add_finding('PERF_ATTENTION', 'Medium', 'Query', q['name'], round(avg, 1), CONFIG['perf_ms_attention'])

perf_df = pd.DataFrame(perf_rows)
show(perf_df)

## 6. Health score, findings e gate

In [ ]:
weights = {'High': 15, 'Medium': 7, 'Low': 2}
score = 100
for f in findings:
    score -= weights.get(f['severity'], 5)
score = max(0, min(100, score))
status = 'Saudavel' if score >= 90 else 'Atencao' if score >= 75 else 'Ruim' if score >= 50 else 'Critico'

corr_pass = int(correctness_df['passed'].sum()) if len(correctness_df) else 0
worst_perf = float(perf_df['avg_ms'].max()) if len(perf_df) else None

summary = pd.DataFrame([{
    'run_ts': run_ts, 'dataset': DATASET, 'workspace': WORKSPACE,
    'model_size_mb': round(total_size_mb, 2),
    'correctness_passed': corr_pass, 'correctness_total': len(correctness_df),
    'perf_worst_avg_ms': worst_perf,
    'findings': len(findings), 'health_score': score, 'health_status': status,
}])
findings_df = pd.DataFrame(findings)

print('Health: {} ({})'.format(score, status))
show(summary)
show(findings_df)

In [ ]:
# Gate: bloqueia a promocao Dev -> Prod se houver finding com severidade em GATE_FAIL_ON
blocking = [f for f in findings if f['severity'] in GATE_FAIL_ON]
if blocking:
    print('X {} problema(s) bloqueante(s):'.format(len(blocking)))
    for b in blocking:
        print('  -', b['rule'], '|', b['object_name'], '|', b.get('details'))
    # Descomente para falhar o notebook (barra o pipeline):
    # raise Exception('Validacao dinamica falhou: findings bloqueantes')
else:
    print('OK - sem problemas bloqueantes')

In [ ]:
# (Opcional) Grava resultados em Delta no lakehouse anexado, para historico/monitoramento
if WRITE_TO_LAKEHOUSE:
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

    def norm(df):
        return df.toDF(*[c.strip().lower().replace(' ', '_') for c in df.columns])

    datasets_to_save = [
        ('dyn_val_perf_results', perf_df),
        ('dyn_val_dax_correctness', correctness_df),
        ('dyn_val_health_summary', summary),
        ('dyn_val_health_findings', findings_df),
    ]
    for name, pdf in datasets_to_save:
        if pdf is not None and len(pdf):
            sdf = norm(spark.createDataFrame(pdf.astype(str)))
            sdf.write.mode('append').format('delta').saveAsTable(name)
    print('Gravado no lakehouse anexado.')
else:
    print('WRITE_TO_LAKEHOUSE = False: nada gravado (apenas exibicao).')